# ⚠️ Important Notices

**Not for production use.** This notebook is sample code for demonstration and educational purposes only. It has not undergone a formal AWS security review. If you plan to build on this sample for production workloads, you are responsible for conducting your own security review, testing, and validation.

**Cost warning.** This notebook uses **two SageMaker real-time endpoints** that incur charges as long as they are running:
- **Quantized endpoint** (`ml.g5.xlarge`): ~$1.41/hr
- **Full-precision endpoint** (`ml.g5.12xlarge`): ~$7.09/hr

**Total: ~$8.50/hr while both endpoints are active.**

Run the **Cleanup** cell at the bottom of this notebook or run `terraform destroy` when you are finished to stop incurring costs.

# Quantized Model Comparison for Amazon SageMaker with Qwen3-VL

Side-by-side comparison of Unsloth's dynamically quantized Qwen3-VL-8B-Instruct (4-bit GGUF via llama.cpp) against the full-precision BF16 variant (via vLLM on SageMaker LMI). Focused on image understanding tasks to showcase vision-language capabilities.

## What Is Quantization?

Large language models store parameters as 16-bit floating-point numbers. An 8B parameter model at BF16 precision needs ~16 GB of GPU memory. **Quantization** reduces this by representing parameters with fewer bits — 4-bit quantization shrinks the same model to ~5 GB.

### Unsloth Dynamic 2.0

Standard quantization applies the same compression uniformly across all layers. [Unsloth Dynamic 2.0](https://unsloth.ai/blog/dynamic-4bit) takes a smarter approach: it **selectively quantizes each layer based on its sensitivity to compression**.

- **Sensitive layers** (early layers, vision projections) are kept at higher precision (16-bit)
- **Less sensitive layers** are compressed more aggressively (4-bit)
- Each model gets a **custom-tailored quantization scheme** using a curated calibration dataset of >1.5M tokens

The result: a model that’s nearly as accurate as the original, at a fraction of the size and cost.

### What We’re Comparing

| | Quantized | Full Precision |
|---|---|---|
| **Model** | unsloth/Qwen3-VL-8B-Instruct-GGUF (Q4_K_M) | Qwen/Qwen3-VL-8B-Instruct (BF16) |
| **Size on disk** | ~6.4 GB | ~16.4 GB |
| **Runtime** | llama.cpp (BYOC container) | vLLM (SageMaker LMI) |
| **Instance** | ml.g5.xlarge (1x A10G, 24 GB) | ml.g5.12xlarge (4x A10G, 96 GB) |
| **Hourly cost** | ~$1.41/hr | ~$7.09/hr |

Both models are the same Qwen3-VL-8B-Instruct architecture — one is just compressed. This notebook sends identical image prompts to both and compares output quality, latency, throughput, and cost.

### Why These Runtimes?

The artifact format drives the runtime choice:
- **GGUF → llama.cpp**: The quantized model is a single GGUF file. llama.cpp is the natural runtime — lightweight, single-GPU, serves GGUF directly via an OpenAI-compatible API.
- **Safetensors (BF16) → vLLM**: The full-precision model stays in standard format. vLLM is built for production GPU serving with batching and high throughput.

### What Is Q4_K_M?

The `Q4_K_M` quantization scheme means: **4-bit** precision, **k-quant** method (groups weights into blocks with per-block scaling factors), **medium** size variant (balances quality vs compression). It’s widely considered the sweet spot for 4-bit quantization.

In [ ]:
import json
import time
import base64
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image as IPImage

from comparison_utils import (
    InferenceResult,
    ComparisonResult,
    PRICING,
    calculate_latency,
    calculate_throughput,
    calculate_cost_per_request,
    compute_average_metrics,
    compute_grouped_averages,
    encode_image,
    build_quantized_payload,
    build_full_precision_payload,
    invoke_endpoint,
    run_comparison,
)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Update these endpoint names to match your Terraform outputs
QUANTIZED_ENDPOINT = "qwen3-vl-8b-quantized"
FULL_PRECISION_ENDPOINT = "qwen3-vl-8b-full-precision"
AWS_REGION = "us-east-2"

# Generation parameters
GENERATION_PARAMS = {
    "max_tokens": 100,
    "temperature": 0.1,
}

# Config dict for run_comparison()
CONFIG = {
    "quantized_endpoint": QUANTIZED_ENDPOINT,
    "full_precision_endpoint": FULL_PRECISION_ENDPOINT,
    "aws_region": AWS_REGION,
}

# Metrics store — accumulates results across all prompts
metrics_store: list[ComparisonResult] = []

print(f"Quantized endpoint:      {QUANTIZED_ENDPOINT}")
print(f"Full-precision endpoint: {FULL_PRECISION_ENDPOINT}")
print(f"Region:                  {AWS_REGION}")

## Deployment Configuration Summary

In [ ]:
config_data = {
    "Property": [
        "HuggingFace Model ID",
        "Quantization",
        "Inference Runtime",
        "SageMaker Instance Type",
        "GPU",
        "Model Size on Disk",
        "Model Family",
    ],
    "Quantized Model": [
        "unsloth/Qwen3-VL-8B-Instruct-GGUF",
        "Unsloth Dynamic 2.0 \u2014 4-bit GGUF (Q4_K_M)",
        "llama.cpp (BYOC)",
        "ml.g5.xlarge",
        "NVIDIA A10G (24 GB)",
        "~6.4 GB (LLM + vision projector)",
        "Qwen3-VL (Vision-Language)",
    ],
    "Full-Precision Model": [
        "Qwen/Qwen3-VL-8B-Instruct",
        "None (BF16 full precision)",
        "vLLM (SageMaker LMI)",
        "ml.g5.12xlarge",
        "NVIDIA A10G x4 (96 GB)",
        "~16.4 GB",
        "Qwen3-VL (Vision-Language)",
    ],
}

df_config = pd.DataFrame(config_data)
display(HTML(
    "<p><em>Both models are Qwen3-VL (Vision-Language) variants that natively support "
    "image understanding. No different model family is required for image tasks.</em></p>"
    + df_config.to_html(index=False)
))

## Image-Based Comparisons

Each cell below sends the same image and prompt to both endpoints, displays the input image, shows the model outputs side by side, and plots per-prompt latency and throughput.

**Note on prompt formatting**: Both endpoints receive semantically identical prompts, but the payload structure differs slightly between the llama.cpp OpenAI-compatible API and the SageMaker LMI/vLLM API. The `comparison_utils.py` module normalizes these differences so both models see the same content. This is important — as noted in the companion blog, prompt formatting mismatches are one of the most common reasons a model appears to perform worse after deployment.

In [ ]:
def display_comparison(result: ComparisonResult):
    """Display side-by-side comparison of model outputs with metrics charts."""
    # Display the prompt
    display(HTML(f"<h3>Prompt: {result.prompt_text}</h3>"))
    
    # Display the input image if this is an image prompt
    if result.image_source:
        display(HTML("<h4>Input Image:</h4>"))
        if result.image_source.startswith("http"):
            display(IPImage(url=result.image_source, width=400))
        else:
            display(IPImage(filename=result.image_source, width=400))
    
    # Side-by-side outputs
    q = result.quantized
    fp = result.full_precision
    
    html = f"""
    <table style="width:100%; border-collapse:collapse; margin:10px 0;">
    <tr style="background:#f0f0f0;">
        <th style="padding:10px; border:1px solid #ddd; width:50%;">{q.model_label}</th>
        <th style="padding:10px; border:1px solid #ddd; width:50%;">{fp.model_label}</th>
    </tr>
    <tr>
        <td style="padding:10px; border:1px solid #ddd; vertical-align:top;">
            {q.error if q.error else q.generated_text}
        </td>
        <td style="padding:10px; border:1px solid #ddd; vertical-align:top;">
            {fp.error if fp.error else fp.generated_text}
        </td>
    </tr>
    </table>
    """
    display(HTML(html))
    
    # Skip charts if both errored
    if q.error and fp.error:
        return
    
    # Per-prompt latency and throughput bar charts
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    labels = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
    colors = ["#2196F3", "#FF9800"]
    
    # Latency chart
    latencies = [q.latency_ms, fp.latency_ms]
    axes[0].bar(labels, latencies, color=colors)
    axes[0].set_ylabel("Latency (ms)")
    axes[0].set_title("End-to-End Latency")
    for i, v in enumerate(latencies):
        axes[0].text(i, v + max(latencies) * 0.02, f"{v:.0f}ms", ha="center", fontsize=10)
    
    # Throughput chart
    throughputs = [q.throughput_tps, fp.throughput_tps]
    axes[1].bar(labels, throughputs, color=colors)
    axes[1].set_ylabel("Tokens/sec")
    axes[1].set_title("Token Generation Throughput")
    for i, v in enumerate(throughputs):
        axes[1].text(i, v + max(throughputs) * 0.02, f"{v:.1f}", ha="center", fontsize=10)
    
    plt.tight_layout()
    plt.show()

### 1. Image Description

In [ ]:
# Download test images locally to avoid rate-limiting from external hosts
import os, urllib.request

os.makedirs('test_images', exist_ok=True)

TEST_IMAGES = {
    'landscape.jpg': 'https://images.unsplash.com/photo-1506744038136-46273834b3fb?w=640',
    'coffee.jpg': 'https://images.unsplash.com/photo-1509042239860-f550ce710b93?w=640',
    'sign.jpg': 'https://images.unsplash.com/photo-1563906267088-b029e7101114?w=640',
    'cat.jpg': 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=640',
    'cityscape.jpg': 'https://images.unsplash.com/photo-1480714378408-67cf0d13bc1b?w=640',
}

for fname, url in TEST_IMAGES.items():
    path = f'test_images/{fname}'
    if not os.path.exists(path):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, path)
    else:
        print(f'{fname} already exists')

print('All test images ready.')

In [ ]:
# Image Description — Describe the contents of an image in detail
IMAGE_1 = "test_images/landscape.jpg"
PROMPT_1 = "Describe this image in one sentence."

result_1 = run_comparison(PROMPT_1, IMAGE_1, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_1)
display_comparison(result_1)

**What to look for:** Both models should identify the same key elements (landscape, mountains, water, etc.). If the quantized model captures the same scene accurately, it demonstrates that Unsloth Dynamic 2.0 preserves visual understanding despite 3x compression. Minor wording differences are expected and don’t indicate quality loss — look for factual errors or hallucinated objects.

### 2. Object Identification

In [ ]:
# Object Identification — Identify and list objects in an image
IMAGE_2 = "test_images/coffee.jpg"
PROMPT_2 = "List the main objects in this image."

result_2 = run_comparison(PROMPT_2, IMAGE_2, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_2)
display_comparison(result_2)

**What to look for:** Compare the object lists. Both models should identify the same core objects. A quantized model that misses objects or invents ones that aren’t there would indicate quality degradation. Matching lists suggest the 4-bit compression preserved the model’s object recognition capability.

### 3. OCR (Text Extraction)

In [ ]:
# OCR — Extract text from an image containing text
IMAGE_3 = "test_images/sign.jpg"
PROMPT_3 = "What text is visible in this image?"

result_3 = run_comparison(PROMPT_3, IMAGE_3, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_3)
display_comparison(result_3)

**What to look for:** OCR is one of the hardest tasks for quantized models because it requires precise character-level recognition. Compare the extracted text word-for-word. Small errors (wrong letters, missing words) are common in heavily quantized models — Unsloth Dynamic 2.0 mitigates this by keeping vision-sensitive layers at higher precision.

### 4. Visual Question Answering

In [ ]:
# Visual QA — Answer a specific question about an image
IMAGE_4 = "test_images/cat.jpg"
PROMPT_4 = "What is the animal doing and where is it?"

result_4 = run_comparison(PROMPT_4, IMAGE_4, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_4)
display_comparison(result_4)

**What to look for:** Visual QA tests reasoning — the model must understand the image and answer a specific question. Both models should agree on the animal type, its activity, and location. Divergent answers here would suggest the quantized model lost some reasoning capability.

### 5. Scene Understanding

In [ ]:
# Scene Understanding — Describe spatial relationships and context
IMAGE_5 = "test_images/cityscape.jpg"
PROMPT_5 = "Describe the layout of this scene in one sentence."

result_5 = run_comparison(PROMPT_5, IMAGE_5, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_5)
display_comparison(result_5)

**What to look for:** Scene understanding requires spatial reasoning — understanding what’s in front, behind, above, below. Both models should describe the same spatial relationships. This is where quantization can sometimes cause subtle errors in relative positioning.

## Text-Only Comparison

A single text-only prompt to compare non-visual performance as a secondary baseline.

### 6. Text Generation (No Image)

In [ ]:
# Text-only comparison — no image input
PROMPT_TEXT = "What is model quantization in one sentence?"

result_text = run_comparison(PROMPT_TEXT, None, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_text)
display_comparison(result_text)

**What to look for:** This text-only prompt serves as a baseline. Without image processing overhead, any latency difference here reflects pure text generation speed. Both models should give accurate, similar definitions — text-only tasks are generally less affected by quantization than vision tasks.

## Metrics Summary

Aggregated performance metrics across all prompts.

In [ ]:
# Average Latency — Overall and by prompt type
q_avg_lat, fp_avg_lat = compute_average_metrics(metrics_store, "latency_ms")
grouped = compute_grouped_averages(metrics_store)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#2196F3", "#FF9800"]

# Overall average latency
labels_overall = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
vals_overall = [q_avg_lat, fp_avg_lat]
axes[0].bar(labels_overall, vals_overall, color=colors)
axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("Average Latency — All Prompts")
for i, v in enumerate(vals_overall):
    axes[0].text(i, v + max(vals_overall) * 0.02, f"{v:.0f}ms", ha="center", fontsize=10)

# Latency by prompt type (grouped bar chart)
prompt_types = sorted(grouped.keys())
x = range(len(prompt_types))
width = 0.35
q_lats = [grouped[pt]["quantized_avg_latency_ms"] for pt in prompt_types]
fp_lats = [grouped[pt]["full_precision_avg_latency_ms"] for pt in prompt_types]

bars1 = axes[1].bar([i - width/2 for i in x], q_lats, width, label="Quantized", color=colors[0])
bars2 = axes[1].bar([i + width/2 for i in x], fp_lats, width, label="Full Precision", color=colors[1])
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Average Latency — By Prompt Type")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([pt.capitalize() for pt in prompt_types])
axes[1].legend()

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{bar.get_height():.0f}", ha="center", fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{bar.get_height():.0f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Average Throughput
q_avg_thr, fp_avg_thr = compute_average_metrics(metrics_store, "throughput_tps")

fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
vals = [q_avg_thr, fp_avg_thr]
colors = ["#2196F3", "#FF9800"]

ax.bar(labels, vals, color=colors)
ax.set_ylabel("Tokens/sec")
ax.set_title("Average Token Generation Throughput — All Prompts")
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.02, f"{v:.1f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

### Cost Comparison

In [ ]:
# Cost Comparison Summary Table
q_hourly = PRICING["ml.g5.xlarge"]
fp_hourly = PRICING["ml.g5.12xlarge"]

q_cost_per_req = calculate_cost_per_request(q_avg_lat, q_hourly)
fp_cost_per_req = calculate_cost_per_request(fp_avg_lat, fp_hourly)

summary_data = {
    "Metric": [
        "Model",
        "Instance Type",
        "Hourly Cost (USD)",
        "Avg Latency (ms)",
        "Avg Throughput (tok/s)",
        "Est. Cost per Request (USD)",
        "Est. Cost per 1K Requests (USD)",
    ],
    "Quantized": [
        "Qwen3-VL-8B \u2014 4-bit GGUF (Q4_K_M)",
        "ml.g5.xlarge",
        f"${q_hourly:.2f}",
        f"{q_avg_lat:.0f}",
        f"{q_avg_thr:.1f}",
        f"${q_cost_per_req:.6f}",
        f"${q_cost_per_req * 1000:.4f}",
    ],
    "Full Precision": [
        "Qwen3-VL-8B \u2014 BF16",
        "ml.g5.12xlarge",
        f"${fp_hourly:.2f}",
        f"{fp_avg_lat:.0f}",
        f"{fp_avg_thr:.1f}",
        f"${fp_cost_per_req:.6f}",
        f"${fp_cost_per_req * 1000:.4f}",
    ],
}

df_summary = pd.DataFrame(summary_data)
display(HTML(df_summary.to_html(index=False)))

## Cleanup

Delete both SageMaker endpoints and their associated resources to stop incurring costs. If any deletion fails, run `terraform destroy` as a fallback.

In [ ]:
# Cleanup — Delete SageMaker endpoints, endpoint configurations, and models
confirm = input("⚠️  This will DELETE both SageMaker endpoints. Type 'yes' to confirm: ")
if confirm.strip().lower() != 'yes':
    print("Cleanup skipped. Endpoints are still running.")
    raise SystemExit("Cleanup cancelled by user")

sm_client = boto3.client("sagemaker", region_name=AWS_REGION)

resources_to_delete = [
    ("endpoint", QUANTIZED_ENDPOINT),
    ("endpoint", FULL_PRECISION_ENDPOINT),
    ("endpoint_config", "qwen3-vl-8b-quantized-config"),
    ("endpoint_config", "qwen3-vl-8b-full-precision-config"),
    ("model", "qwen3-vl-8b-quantized"),
    ("model", "qwen3-vl-8b-full-precision"),
]

for resource_type, resource_name in resources_to_delete:
    try:
        if resource_type == "endpoint":
            sm_client.delete_endpoint(EndpointName=resource_name)
        elif resource_type == "endpoint_config":
            sm_client.delete_endpoint_config(EndpointConfigName=resource_name)
        elif resource_type == "model":
            sm_client.delete_model(ModelName=resource_name)
        print(f"✅ Deleted {resource_type}: {resource_name}")
    except sm_client.exceptions.ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "ValidationException" and "Could not find" in str(e):
            print(f"ℹ️  {resource_type} '{resource_name}' already deleted.")
        else:
            print(f"❌ Failed to delete {resource_type} '{resource_name}': {e}")
            print("   Fallback: run 'terraform destroy' from the terraform/ directory.")
    except Exception as e:
        print(f"❌ Failed to delete {resource_type} '{resource_name}': {e}")
        print("   Fallback: run 'terraform destroy' from the terraform/ directory.")

print("\nCleanup complete. If any deletions failed, run 'terraform destroy' as a fallback.")